In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import torch.nn as nn

In [3]:
import pandas as pd
from next_sequence.utils import create_bow

df = pd.DataFrame({
    'id': [1,2,3],
    'item': [['a','a','b'], ['a','b'], ['a','a','a','a','b']],
    'age': [[25,26,27], [35,37], [41,41,42,42,55]],
    'price': [[100, 110, 120], [1000, 1500], [1000, 1000, 1000, 1000, 2500]],
})

bow = create_bow(df['item'].tolist())
df['item'] = df['item'].apply(lambda x: [bow[_x] for _x in x])
# len(bow), bow
df

,id,item,age,price
0,1,"[2, 2, 1]","[25, 26, 27]","[100, 110, 120]"
1,2,"[2, 1]","[35, 37]","[1000, 1500]"
2,3,"[2, 2, 2, 2, 1]","[41, 41, 42, 42, 55]","[1000, 1000, 1000, 1000, 2500]"


In [4]:
# data collate and batch
# all custom collate_fn here will be changed project by project
def _pack_xy(b, dtype):
    return torch.tensor(b, dtype=dtype)

def collate_common(batch, inference=False):
    if inference:
        return [_pack_xy(b, torch.long) for b in batch], None
    
    x = [_pack_xy(b[:-1], torch.long) for b in batch]
    y = [_pack_xy(b[-1], torch.long) for b in batch]
    return x, y

def collate_delta(batch, inference=False):
    if inference:
        return [_pack_xy(b, torch.float) for b in batch], None

    x = [_pack_xy(b[:-1], torch.float) for b in batch]
    y = [_pack_xy(b[-1]-b[-2], torch.float) for b in batch]
    return x, y

def collate_delta_bin(batch, inference=False):
    if inference:
        return [_pack_xy(b, torch.long) for b in batch], None

    def logic(b):
        if b <= 1:
            return 1
        if b <= 3:
            return 2
        return 3
    # if embedding is linear, x dtype must be float only
    x = [_pack_xy(b[:-1], torch.float) for b in batch]
    y = [_pack_xy(logic(b[-1]-b[-2]), torch.long) for b in batch]
    return x, y

In [5]:
from next_sequence.model import NextSequenceConfig, NextSequenceModel
from next_sequence.core import Feature, TaskType
from next_sequence.fusion_registry import FusionStrategy

features = [
    Feature(name='item', task=TaskType.CLASSIFICATION, vocab_size=len(bow)),
    Feature(name='age', task=TaskType.CLASSIFICATION, vocab_size=4, is_linear=True),
    Feature(name='price', task=TaskType.REGRESSION, is_linear=True),
]
configs = NextSequenceConfig(
    features=features,
    max_len=4,
    emb_dim=128,
    # fusion_dim=128,
    fusion_dim=128*2,
    # fusion_strategy=FusionStrategy.SUM,
    fusion_strategy=FusionStrategy.CONCAT,
)
model = NextSequenceModel(configs=configs)

In [6]:
model

NextSequenceModel(
  (embedding): EmbeddingModel(
    (embedder): ModuleDict(
      (item): Embedding(3, 128, padding_idx=0)
      (age): Linear(in_features=1, out_features=128, bias=True)
      (price): Linear(in_features=1, out_features=128, bias=True)
    )
  )
  (fusion): ConcatFusion(
    (proj): Linear(in_features=384, out_features=256, bias=True)
  )
  (prepend_cls): PrependCLS()
  (rope): RoPE()
  (backbone): EncoderCLSBackbone(
    (embedding): EmbeddingModel(
      (embedder): ModuleDict(
        (item): Embedding(3, 128, padding_idx=0)
        (age): Linear(in_features=1, out_features=128, bias=True)
        (price): Linear(in_features=1, out_features=128, bias=True)
      )
    )
    (fusion): ConcatFusion(
      (proj): Linear(in_features=384, out_features=256, bias=True)
    )
    (prepend_cls): PrependCLS()
    (rope): RoPE()
    (layers): ModuleList(
      (0-1): 2 x EncoderBlock(
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        

In [16]:
from next_sequence.core import CoreDataset
from torch.utils.data import DataLoader
from next_sequence.collate_fn import CustomCollate
funcs = {
    'item': collate_common,
    'age': collate_delta_bin,
    'price': collate_delta
}
collate_fn = CustomCollate(features, 'item', funcs, inference=False)
samples = df.to_dict(orient='records')
dataset = CoreDataset(samples)
batch = dataset[:]
batch = collate_fn(batch)
batch

{'X': defaultdict(list,
             {'item': tensor([[2, 2, 0, 0],
                      [2, 0, 0, 0],
                      [2, 2, 2, 2]]),
              'age': tensor([[[25.],
                       [26.],
                       [ 0.],
                       [ 0.]],
              
                      [[35.],
                       [ 0.],
                       [ 0.],
                       [ 0.]],
              
                      [[41.],
                       [41.],
                       [42.],
                       [42.]]]),
              'price': tensor([[[ 100.],
                       [ 110.],
                       [   0.],
                       [   0.]],
              
                      [[1000.],
                       [   0.],
                       [   0.],
                       [   0.]],
              
                      [[1000.],
                       [1000.],
                       [1000.],
                       [1000.]]])}),
 'Y': defaultdict(list,
  

In [17]:
model.eval()
with torch.no_grad():
    predictions = model(batch['X'], batch['mask'])
predictions

{'item': tensor([[-0.0139,  0.4888, -0.3057],
         [ 0.0381,  0.4381, -0.2223],
         [-0.0230,  0.3914, -0.0877]]),
 'age': tensor([[-0.2570,  0.3217, -0.2687,  0.7385],
         [-0.1843,  0.3526, -0.2688,  0.6693],
         [-0.5220,  0.2777, -0.2516,  1.0435]]),
 'price': tensor([[-0.2652],
         [-0.1099],
         [-0.1606]])}

In [18]:
model.embedding.d_model, model.fusion.d_model, model.fusion.d_out

(128, 384, 256)

In [19]:
from next_sequence.loss_fn import calculate_loss, UncertaintyWeightedLoss

criterion = UncertaintyWeightedLoss(features)

In [22]:
losses = calculate_loss(batch['Y'], predictions, features)
losses, sum(losses.values())

({'item': tensor(0.7760), 'age': tensor(1.2763), 'price': tensor(833565.7500)},
 tensor(833567.8125))

In [21]:
criterion(losses)

tensor(416784.9375, grad_fn=<SumBackward0>)